In [1]:
# Cell 1: Install required libraries (agar nahi hai toh)
!pip install pandas openpyxl sqlalchemy

# Cell 2: Import libraries
import pandas as pd
import sqlite3
import os

print("✅ Libraries imported successfully!")


[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


✅ Libraries imported successfully!


In [2]:
# Cell 3: Load cleaned dataset
df = pd.read_excel('Cleaned_Dataset.xlsx')
print(f"✅ Loaded: {df.shape[0]} rows, {df.shape[1]} columns")
df.head()

✅ Loaded: 1200 rows, 14 columns


,OrderID,Date,CustomerID,Product,Quantity,UnitPrice,ShippingAddress,PaymentMethod,OrderStatus,TrackingNumber,ItemsInCart,CouponCode,ReferralSource,TotalPrice
0,ORD200000,2023-01-04,C72649,Monitor,5,570.62,928 Main St,Debit Card,Shipped,TRK37947903,7,SAVE10,Instagram,2853.10
1,ORD200001,2024-08-23,C75739,Phone,2,151.35,823 Main St,Online,Shipped,TRK91186779,3,SAVE10,Referral,302.70
2,ORD200002,2024-02-27,C81728,Tablet,5,550.68,512 Main St,Credit Card,Cancelled,TRK42903982,8,FREESHIP,Email,2753.40
3,ORD200003,2023-10-15,C33540,Chair,1,273.19,275 Main St,Debit Card,Returned,TRK62788070,5,SAVE10,Facebook,273.19
4,ORD200004,2025-05-08,C81840,Printer,4,626.01,668 Main St,Online,Delivered,TRK29241424,8,SAVE10,Email,2504.04


In [4]:
# Cell 4: Create SQLite database
conn = sqlite3.connect('bridal_ease.db')

# Save dataframe to SQL table
df.to_sql('orders', conn, if_exists='replace', index=False)

print("✅ Database 'Project 3.db' created!")
print("✅ Table 'orders' created with 1200 records")

# Verify
cursor = conn.cursor()
cursor.execute("SELECT COUNT(*) FROM orders")
count = cursor.fetchone()[0]
print(f"📊 Total records in database: {count}")

✅ Database 'Project 3.db' created!
✅ Table 'orders' created with 1200 records
📊 Total records in database: 1200


In [5]:
# Cell 5: Check table schema
cursor.execute("PRAGMA table_info(orders)")
columns = cursor.fetchall()
print("📋 Table Structure:")
for col in columns:
    print(f"   - {col[1]} ({col[2]})")

📋 Table Structure:
   - OrderID (TEXT)
   - Date (TIMESTAMP)
   - CustomerID (TEXT)
   - Product (TEXT)
   - Quantity (INTEGER)
   - UnitPrice (REAL)
   - ShippingAddress (TEXT)
   - PaymentMethod (TEXT)
   - OrderStatus (TEXT)
   - TrackingNumber (TEXT)
   - ItemsInCart (INTEGER)
   - CouponCode (TEXT)
   - ReferralSource (TEXT)
   - TotalPrice (REAL)


In [6]:
# Cell 6: Simple SELECT
query1 = "SELECT * FROM orders LIMIT 10;"
result1 = pd.read_sql_query(query1, conn)
print("📊 First 10 rows:")
result1

📊 First 10 rows:


,OrderID,Date,CustomerID,Product,Quantity,UnitPrice,ShippingAddress,PaymentMethod,OrderStatus,TrackingNumber,ItemsInCart,CouponCode,ReferralSource,TotalPrice
0,ORD200000,2023-01-04 00:00:00,C72649,Monitor,5,570.62,928 Main St,Debit Card,Shipped,TRK37947903,7,SAVE10,Instagram,2853.10
1,ORD200001,2024-08-23 00:00:00,C75739,Phone,2,151.35,823 Main St,Online,Shipped,TRK91186779,3,SAVE10,Referral,302.70
2,ORD200002,2024-02-27 00:00:00,C81728,Tablet,5,550.68,512 Main St,Credit Card,Cancelled,TRK42903982,8,FREESHIP,Email,2753.40
3,ORD200003,2023-10-15 00:00:00,C33540,Chair,1,273.19,275 Main St,Debit Card,Returned,TRK62788070,5,SAVE10,Facebook,273.19
4,ORD200004,2025-05-08 00:00:00,C81840,Printer,4,626.01,668 Main St,Online,Delivered,TRK29241424,8,SAVE10,Email,2504.04
5,ORD200005,2023-10-23 00:00:00,C37249,Phone,2,245.86,934 Main St,Credit Card,Shipped,TRK72976927,4,SAVE10,Instagram,491.72
6,ORD200006,2025-06-17 00:00:00,C83492,Laptop,1,664.42,986 Main St,Gift Card,Returned,TRK96417362,6,SAVE10,Facebook,664.42
7,ORD200007,2023-05-12 00:00:00,C41460,Monitor,5,149.55,706 Main St,Cash,Shipped,TRK78809193,9,FREESHIP,Facebook,747.75
8,ORD200008,2025-04-02 00:00:00,C26817,Phone,2,134.28,904 Main St,Gift Card,Cancelled,TRK61042692,2,No Coupon,Email,268.56
9,ORD200009,2023-11-21 00:00:00,C31946,Desk,4,509.38,102 Main St,Credit Card,Shipped,TRK33478363,6,SAVE10,Google,2037.52


In [7]:
# Cell 7: Orders with TotalPrice > 2000
query2 = """
SELECT OrderID, Product, TotalPrice 
FROM orders 
WHERE TotalPrice > 2000
ORDER BY TotalPrice DESC;
"""
result2 = pd.read_sql_query(query2, conn)
print(f"💰 Orders with TotalPrice > 2000: {len(result2)} orders")
result2.head(10)

💰 Orders with TotalPrice > 2000: 180 orders


,OrderID,Product,TotalPrice
0,ORD200789,Tablet,3456.40
1,ORD201122,Monitor,3390.95
2,ORD200632,Laptop,3390.80
3,ORD200469,Chair,3384.90
4,ORD200328,Tablet,3370.20
5,ORD200107,Printer,3353.75
6,ORD200326,Laptop,3352.40
7,ORD201065,Printer,3334.00
8,ORD201031,Phone,3322.55
9,ORD200463,Laptop,3313.90


In [8]:
# Cell 8: Cancelled orders
query3 = """
SELECT OrderID, Product, OrderStatus, TotalPrice
FROM orders 
WHERE OrderStatus = 'Cancelled'
LIMIT 10;
"""
result3 = pd.read_sql_query(query3, conn)
print("❌ Cancelled Orders:")
result3

❌ Cancelled Orders:


,OrderID,Product,OrderStatus,TotalPrice
0,ORD200002,Tablet,Cancelled,2753.40
1,ORD200008,Phone,Cancelled,268.56
2,ORD200023,Phone,Cancelled,1093.00
3,ORD200033,Chair,Cancelled,162.77
4,ORD200042,Printer,Cancelled,303.32
5,ORD200043,Chair,Cancelled,1497.63
6,ORD200050,Tablet,Cancelled,179.22
7,ORD200055,Printer,Cancelled,1033.20
8,ORD200056,Tablet,Cancelled,1437.63
9,ORD200058,Phone,Cancelled,1898.76


In [9]:
# Cell 9: Orders from 2024
query4 = """
SELECT OrderID, Product, Date, TotalPrice
FROM orders 
WHERE Date LIKE '2024%'
LIMIT 10;
"""
result4 = pd.read_sql_query(query4, conn)
print("📅 Orders from 2024:")
result4

📅 Orders from 2024:


,OrderID,Product,Date,TotalPrice
0,ORD200001,Phone,2024-08-23 00:00:00,302.70
1,ORD200002,Tablet,2024-02-27 00:00:00,2753.40
2,ORD200011,Monitor,2024-02-17 00:00:00,147.42
3,ORD200012,Monitor,2024-10-15 00:00:00,361.00
4,ORD200017,Tablet,2024-03-02 00:00:00,423.40
5,ORD200021,Monitor,2024-11-17 00:00:00,1371.80
6,ORD200024,Laptop,2024-10-30 00:00:00,461.90
7,ORD200025,Monitor,2024-07-24 00:00:00,399.98
8,ORD200026,Laptop,2024-03-06 00:00:00,297.75
9,ORD200027,Printer,2024-02-22 00:00:00,472.74


In [10]:
# Cell 10: Instagram referral with high value
query5 = """
SELECT OrderID, ReferralSource, TotalPrice
FROM orders 
WHERE ReferralSource = 'Instagram' AND TotalPrice > 1500
ORDER BY TotalPrice DESC;
"""
result5 = pd.read_sql_query(query5, conn)
print(f"📱 Instagram orders > Rs.1500: {len(result5)} orders")
result5

📱 Instagram orders > Rs.1500: 69 orders


,OrderID,ReferralSource,TotalPrice
0,ORD200107,Instagram,3353.75
1,ORD200463,Instagram,3313.90
2,ORD200367,Instagram,3293.85
3,ORD200010,Instagram,3129.85
4,ORD200450,Instagram,3075.50
...,...,...,...
64,ORD200442,Instagram,1574.70
65,ORD200057,Instagram,1571.35
66,ORD200228,Instagram,1543.68
67,ORD200794,Instagram,1539.40


In [11]:
# Cell 11: Product-wise revenue
query6 = """
SELECT Product, 
       SUM(TotalPrice) as TotalRevenue,
       COUNT(*) as OrderCount
FROM orders
GROUP BY Product
ORDER BY TotalRevenue DESC;
"""
result6 = pd.read_sql_query(query6, conn)
print("🏆 Product-wise Revenue:")
result6

🏆 Product-wise Revenue:


,Product,TotalRevenue,OrderCount
0,Chair,195620.11,178
1,Printer,195612.61,181
2,Laptop,192126.56,173
3,Tablet,186568.95,179
4,Monitor,175651.41,163
5,Desk,167459.93,170
6,Phone,151722.39,156


In [12]:
# Cell 12: Product-wise quantity sold
query7 = """
SELECT Product, 
       SUM(Quantity) as TotalQuantity
FROM orders
GROUP BY Product
ORDER BY TotalQuantity DESC;
"""
result7 = pd.read_sql_query(query7, conn)
print("📦 Product-wise Quantity Sold:")
result7


📦 Product-wise Quantity Sold:


,Product,TotalQuantity
0,Chair,562
1,Printer,542
2,Laptop,535
3,Desk,508
4,Tablet,497
5,Monitor,480
6,Phone,411


In [13]:
# Cell 13: Payment method analysis
query8 = """
SELECT PaymentMethod, 
       COUNT(*) as OrderCount,
       AVG(TotalPrice) as AvgOrderValue,
       SUM(TotalPrice) as TotalRevenue
FROM orders
GROUP BY PaymentMethod
ORDER BY AvgOrderValue DESC;
"""
result8 = pd.read_sql_query(query8, conn)
print("💳 Payment Method Analysis:")
result8

💳 Payment Method Analysis:


,PaymentMethod,OrderCount,AvgOrderValue,TotalRevenue
0,Credit Card,234,1127.553974,263847.63
1,Gift Card,230,1070.973565,246323.92
2,Cash,246,1056.041829,259786.29
3,Online,258,1017.220698,262442.94
4,Debit Card,232,1001.556810,232361.18


In [14]:
# Cell 14: Order status counts
query9 = """
SELECT OrderStatus, 
       COUNT(*) as OrderCount,
       ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM orders), 1) as Percentage
FROM orders
GROUP BY OrderStatus
ORDER BY OrderCount DESC;
"""
result9 = pd.read_sql_query(query9, conn)
print("📊 Order Status Distribution:")
result9

📊 Order Status Distribution:


,OrderStatus,OrderCount,Percentage
0,Cancelled,250,20.8
1,Returned,247,20.6
2,Pending,237,19.8
3,Shipped,235,19.6
4,Delivered,231,19.3


In [15]:
# Cell 15: Referral source performance
query10 = """
SELECT ReferralSource, 
       COUNT(*) as OrderCount,
       SUM(TotalPrice) as TotalRevenue
FROM orders
GROUP BY ReferralSource
ORDER BY TotalRevenue DESC;
"""
result10 = pd.read_sql_query(query10, conn)
print("📢 Referral Source Analysis:")
result10

📢 Referral Source Analysis:


,ReferralSource,OrderCount,TotalRevenue
0,Instagram,259,275285.45
1,Email,250,261808.55
2,Google,241,250441.48
3,Facebook,228,250410.90
4,Referral,222,226815.58


In [16]:
# Cell 16: Products with revenue > 1,50,000
query11 = """
SELECT Product, 
       SUM(TotalPrice) as TotalRevenue
FROM orders
GROUP BY Product
HAVING SUM(TotalPrice) > 150000
ORDER BY TotalRevenue DESC;
"""
result11 = pd.read_sql_query(query11, conn)
print("🏆 Products with Revenue > Rs.1,50,000:")
result11

🏆 Products with Revenue > Rs.1,50,000:


,Product,TotalRevenue
0,Chair,195620.11
1,Printer,195612.61
2,Laptop,192126.56
3,Tablet,186568.95
4,Monitor,175651.41
5,Desk,167459.93
6,Phone,151722.39


In [17]:
# Cell 17: Revenue by year
query12 = """
SELECT SUBSTR(Date, 1, 4) as Year,
       COUNT(*) as OrderCount,
       SUM(TotalPrice) as TotalRevenue,
       ROUND(AVG(TotalPrice), 2) as AvgOrderValue
FROM orders
GROUP BY Year
ORDER BY Year;
"""
result12 = pd.read_sql_query(query12, conn)
print("📅 Year-wise Analysis:")
result12

📅 Year-wise Analysis:


,Year,OrderCount,TotalRevenue,AvgOrderValue
0,2023,510,552643.24,1083.61
1,2024,459,480235.87,1046.27
2,2025,231,231882.85,1003.82


In [18]:
# Cell 18: Top coupon codes
query13 = """
SELECT CouponCode, 
       COUNT(*) as UsageCount
FROM orders
WHERE CouponCode != 'No Coupon'
GROUP BY CouponCode
ORDER BY UsageCount DESC
LIMIT 5;
"""
result13 = pd.read_sql_query(query13, conn)
print("🎟️ Top 5 Coupon Codes:")
result13

🎟️ Top 5 Coupon Codes:


,CouponCode,UsageCount
0,FREESHIP,313
1,WINTER15,292
2,SAVE10,286


In [19]:
# Cell 19: Product and order status combination
query14 = """
SELECT Product, OrderStatus, 
       COUNT(*) as OrderCount,
       SUM(TotalPrice) as TotalRevenue
FROM orders
GROUP BY Product, OrderStatus
ORDER BY Product, TotalRevenue DESC
LIMIT 15;
"""
result14 = pd.read_sql_query(query14, conn)
print("📦 Product × Status Analysis:")
result14

📦 Product × Status Analysis:


,Product,OrderStatus,OrderCount,TotalRevenue
0,Chair,Cancelled,45,48660.98
1,Chair,Pending,41,48504.70
2,Chair,Shipped,31,40350.44
3,Chair,Delivered,33,31465.83
4,Chair,Returned,28,26638.16
5,Desk,Pending,38,41390.08
6,Desk,Cancelled,35,39587.69
7,Desk,Shipped,33,33424.89
8,Desk,Returned,32,28831.49
9,Desk,Delivered,32,24225.78


In [20]:
# Cell 20: Average quantity per order by product
query15 = """
SELECT Product, 
       AVG(Quantity) as AvgQuantity,
       MIN(Quantity) as MinQuantity,
       MAX(Quantity) as MaxQuantity
FROM orders
GROUP BY Product
ORDER BY AvgQuantity DESC;
"""
result15 = pd.read_sql_query(query15, conn)
print("📊 Average Quantity by Product:")
result15

📊 Average Quantity by Product:


,Product,AvgQuantity,MinQuantity,MaxQuantity
0,Chair,3.157303,1,5
1,Laptop,3.092486,1,5
2,Printer,2.994475,1,5
3,Desk,2.988235,1,5
4,Monitor,2.944785,1,5
5,Tablet,2.776536,1,5
6,Phone,2.634615,1,5


In [21]:
# Cell 21: High value orders (outliers)
query16 = """
SELECT OrderID, Product, TotalPrice
FROM orders
WHERE TotalPrice > 3300
ORDER BY TotalPrice DESC;
"""
result16 = pd.read_sql_query(query16, conn)
print(f"⚠️ Outlier orders (TotalPrice > 3300): {len(result16)} orders")
result16

⚠️ Outlier orders (TotalPrice > 3300): 10 orders


,OrderID,Product,TotalPrice
0,ORD200789,Tablet,3456.40
1,ORD201122,Monitor,3390.95
2,ORD200632,Laptop,3390.80
3,ORD200469,Chair,3384.90
4,ORD200328,Tablet,3370.20
5,ORD200107,Printer,3353.75
6,ORD200326,Laptop,3352.40
7,ORD201065,Printer,3334.00
8,ORD201031,Phone,3322.55
9,ORD200463,Laptop,3313.90


In [22]:
# Cell 22: Monthly revenue for 2024
query17 = """
SELECT SUBSTR(Date, 6, 2) as Month,
       COUNT(*) as OrderCount,
       SUM(TotalPrice) as TotalRevenue
FROM orders
WHERE Date LIKE '2024%'
GROUP BY Month
ORDER BY Month;
"""
result17 = pd.read_sql_query(query17, conn)
print("📈 Monthly Revenue Trend - 2024:")
result17

📈 Monthly Revenue Trend - 2024:


,Month,OrderCount,TotalRevenue
0,01,32,38528.08
1,02,32,36909.57
2,03,36,36030.90
3,04,50,49613.14
4,05,34,27909.11
5,06,53,68068.54
6,07,43,42963.98
7,08,28,31991.07
8,09,44,39794.98
9,10,31,37226.97


In [23]:
# Cell 23: ItemsInCart vs TotalPrice
query18 = """
SELECT ItemsInCart,
       COUNT(*) as OrderCount,
       AVG(TotalPrice) as AvgTotalPrice,
       MIN(TotalPrice) as MinPrice,
       MAX(TotalPrice) as MaxPrice
FROM orders
GROUP BY ItemsInCart
ORDER BY ItemsInCart;
"""
result18 = pd.read_sql_query(query18, conn)
print("🛒 ItemsInCart Analysis:")
result18

🛒 ItemsInCart Analysis:


,ItemsInCart,OrderCount,AvgTotalPrice,MinPrice,MaxPrice
0,1,50,368.680200,24.48,687.89
1,2,78,551.362051,17.24,1395.86
2,3,123,722.423008,53.00,2078.13
3,4,163,831.135460,18.20,2786.84
4,5,191,1055.587382,14.06,3384.90
5,6,190,1119.572368,11.39,3223.20
6,7,151,1157.716291,57.10,3390.80
7,8,129,1488.234574,48.40,3390.95
8,9,79,1489.458101,58.76,3313.90
9,10,46,1743.218478,107.70,3456.40


In [24]:
# Cell 24: Export all query results to Excel
with pd.ExcelWriter('SQL_Query_Results.xlsx') as writer:
    result6.to_excel(writer, sheet_name='Revenue_by_Product', index=False)
    result7.to_excel(writer, sheet_name='Quantity_by_Product', index=False)
    result8.to_excel(writer, sheet_name='Payment_Methods', index=False)
    result9.to_excel(writer, sheet_name='Order_Status', index=False)
    result10.to_excel(writer, sheet_name='Referral_Sources', index=False)
    result11.to_excel(writer, sheet_name='High_Revenue_Products', index=False)
    result12.to_excel(writer, sheet_name='Yearly_Analysis', index=False)
    result13.to_excel(writer, sheet_name='Top_Coupons', index=False)
    result16.to_excel(writer, sheet_name='Outliers', index=False)
    result17.to_excel(writer, sheet_name='Monthly_Trend', index=False)

print("✅ All results saved to 'SQL_Query_Results.xlsx'")

✅ All results saved to 'SQL_Query_Results.xlsx'


In [25]:
# Cell 25: Save all SQL queries to .sql file
sql_queries = """
-- PROJECT 3: SQL DATA ANALYSIS
-- Analyst: [Your Name]
-- Date: May 2026

-- Query 1: Orders with TotalPrice > 2000
SELECT OrderID, Product, TotalPrice 
FROM orders 
WHERE TotalPrice > 2000
ORDER BY TotalPrice DESC;

-- Query 2: Cancelled Orders
SELECT OrderID, Product, OrderStatus, TotalPrice
FROM orders 
WHERE OrderStatus = 'Cancelled';

-- Query 3: Product-wise Revenue
SELECT Product, SUM(TotalPrice) as TotalRevenue
FROM orders
GROUP BY Product
ORDER BY TotalRevenue DESC;

-- Query 4: Order Status Distribution
SELECT OrderStatus, COUNT(*) as OrderCount
FROM orders
GROUP BY OrderStatus
ORDER BY OrderCount DESC;

-- Query 5: Referral Source Analysis
SELECT ReferralSource, COUNT(*) as OrderCount, SUM(TotalPrice) as TotalRevenue
FROM orders
GROUP BY ReferralSource
ORDER BY TotalRevenue DESC;

-- Query 6: Year-wise Revenue
SELECT SUBSTR(Date, 1, 4) as Year, SUM(TotalPrice) as TotalRevenue
FROM orders
GROUP BY Year
ORDER BY Year;

-- Query 7: Top Coupon Codes
SELECT CouponCode, COUNT(*) as UsageCount
FROM orders
WHERE CouponCode != 'No Coupon'
GROUP BY CouponCode
ORDER BY UsageCount DESC
LIMIT 5;

-- Query 8: Outlier Detection
SELECT OrderID, Product, TotalPrice
FROM orders
WHERE TotalPrice > 3300
ORDER BY TotalPrice DESC;

-- Query 9: Monthly Revenue Trend 2024
SELECT SUBSTR(Date, 6, 2) as Month, SUM(TotalPrice) as TotalRevenue
FROM orders
WHERE Date LIKE '2024%'
GROUP BY Month
ORDER BY Month;

-- Query 10: Products with Revenue > 150000
SELECT Product, SUM(TotalPrice) as TotalRevenue
FROM orders
GROUP BY Product
HAVING SUM(TotalPrice) > 150000;
"""

with open('SQL_Queries.sql', 'w') as f:
    f.write(sql_queries)

print("✅ SQL queries saved to 'SQL_Queries.sql'")

✅ SQL queries saved to 'SQL_Queries.sql'


In [26]:
# Cell 26: Close connection
conn.close()
print("✅ Database connection closed")

✅ Database connection closed


In [28]:
# Cell 27: Final summary
print("="*60)
print("✅ PROJECT 3 - SQL DATA ANALYSIS COMPLETED!")
print("="*60)

print("\n📁 FILES GENERATED:")
print("   1. bridal_ease.db - SQLite database")
print("   2. SQL_Query_Results.xlsx - All query outputs")
print("   3. SQL_Queries.sql - All SQL queries")

print("\n📊 QUERIES WRITTEN: 18")
print("   • Basic SELECT: 5 queries")
print("   • GROUP BY: 6 queries")
print("   • HAVING: 1 query")
print("   • Aggregations: 6 queries")



✅ PROJECT 3 - SQL DATA ANALYSIS COMPLETED!

📁 FILES GENERATED:
   1. bridal_ease.db - SQLite database
   2. SQL_Query_Results.xlsx - All query outputs
   3. SQL_Queries.sql - All SQL queries

📊 QUERIES WRITTEN: 18
   • Basic SELECT: 5 queries
   • GROUP BY: 6 queries
   • HAVING: 1 query
   • Aggregations: 6 queries


In [30]:
import sqlite3
import pandas as pd

# Connect
conn = sqlite3.connect('Project 3.db')

# Poori table dekho
df_full = pd.read_sql_query("SELECT * FROM orders;", conn)
print(f"Total rows: {len(df_full)}")
df_full.head(20)  # Pehle 20 rows

# Close
conn.close()

Total rows: 1200
